# v32 参照投票ゲート付き hedge — OOF検証(真値なしルール、test完全再現可能)
ルール: 井戸ごとに **d_ref = median(pf_ref − blend予測)** を観測(pf_ref = nbr/self のPF軌跡)。
`GAP_MIN <= |d_ref| <= GAP_MAX` なら「datumが参照と割れている」= ゲート開 → `out = blend + λ·d_ref`(井戸一律シフト)。
- 調整は **λ / GAP_MIN / 参照ソース の3つだけ・粗く**。per-well調整なし。
- 判定: OOF overall(基準 6.8711=combo blend) と wells>10。**硬い核27本での回収 vs 正常井戸の破壊**を分けて見る。
- ここで勝てば submit に同じ数行を実装(入力は test_df の pf 列のみ)。


In [ ]:
# ===== ロード: blend OOF + 参照PFの井戸別オフセット =====
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "Meiryo"; matplotlib.rcParams["axes.unicode_minus"] = False

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
V28_ART = PROJECT / "v28" / "artifacts_v28"
V29_ART = PROJECT / "v29" / "artifacts_v29"
V29B    = PROJECT / "v29" / "artifacts_v29b"
ART = PROJECT / "v32" / "artifacts_v32"; ART.mkdir(parents=True, exist_ok=True)

tbl = pq.read_table(V28_ART / "data" / "train.parquet",
    columns=["well", "target", "last_known_tvt", "md_since", "pf_ancc",
             "pf_self_mean", "pf_nbr_mean", "pf_nbr_has"])
g   = tbl.column("well").to_pandas().to_numpy()
y   = tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
lk  = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pfa = tbl.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
pfs = tbl.column("pf_self_mean").to_numpy(zero_copy_only=False).astype(np.float64)
pfn = tbl.column("pf_nbr_mean").to_numpy(zero_copy_only=False).astype(np.float64)
has = tbl.column("pf_nbr_has").to_numpy(zero_copy_only=False).astype(np.float64)
tvt_true = lk + y

gbm = np.load(V29_ART / "gbm_oof.npy")
nnz = np.load(V29B / "nn_oof_combo.npz")
nn_ens = np.mean([nnz[k] for k in nnz.files], axis=0)
w_nn = float(json.loads((V29B / "results.json").read_text())["combo"]["blend_w"])
res_b = (1 - w_nn) * gbm + w_nn * nn_ens
pp = json.loads((V28_ART / "pp_params.json").read_text())
d_ = res_b * (1 - pp["w_pf"]) + (pfa - lk) * pp["w_pf"]
d_ = d_ * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(pp["tau"]))) * pp.get("alpha", 1.0)
blend = lk + d_

wid = pd.unique(g)
rows_of = {w: np.where(g == w)[0] for w in wid}
def rmse(e):
    e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e ** 2))) if e.size else np.nan
base_overall = rmse(blend - tvt_true)
per_base = {w: rmse((blend - tvt_true)[rows_of[w]]) for w in wid}
print(f"blend OOF overall = {base_overall:.4f}(基準)")

# 井戸別の観測量(真値不使用): 参照PFと blend の datum差
OBS = {}
for w in wid:
    r = rows_of[w]
    d_nbr = float(np.nanmedian((pfn - blend)[r])) if has[r].mean() > 0.5 else np.nan
    d_self = float(np.nanmedian((pfs - blend)[r]))
    d_tw = float(np.nanmedian((pfa - blend)[r]))
    OBS[w] = dict(d_nbr=d_nbr, d_self=d_self, d_tw=d_tw)
HARD27 = [w for w in pd.read_csv(ART / "hardcore_diag.csv", index_col=0).query(
    "rmse_prior >= 10 and abs(bias) > 8").index if w in rows_of]
print(f"wells {len(wid)} | 硬い核(bias) {len(HARD27)}本")


In [ ]:
# ===== ルールのグリッド評価(λ × GAP_MIN × 参照ソース) =====
GAP_MAX = 45.0
def d_ref_of(w, src):
    o = OBS[w]
    if src == "nbr":  return o["d_nbr"]
    if src == "self": return o["d_self"]
    if src == "nbr_or_self":
        return o["d_nbr"] if np.isfinite(o["d_nbr"]) else o["d_self"]
    if src == "mean":
        v = [x for x in (o["d_nbr"], o["d_self"]) if np.isfinite(x)]
        return float(np.mean(v)) if v else np.nan
    raise ValueError(src)

def apply_rule(src, gap_min, lam):
    out = blend.copy(); n_gate = 0; gated = []
    for w in wid:
        d = d_ref_of(w, src)
        if np.isfinite(d) and gap_min <= abs(d) <= GAP_MAX:
            out[rows_of[w]] += lam * d; n_gate += 1; gated.append(w)
    return out, n_gate, gated

res = []
for src in ["nbr", "self", "nbr_or_self", "mean"]:
    for gap in [6.0, 8.0, 12.0]:
        for lam in [0.1, 0.2, 0.3, 0.4, 0.5]:
            out, n_gate, gated = apply_rule(src, gap, lam)
            ov = rmse(out - tvt_true)
            per_new = {w: rmse((out - tvt_true)[rows_of[w]]) for w in gated}
            hurt = sum(1 for w in gated if per_new[w] > per_base[w] + 0.5)
            help_ = sum(1 for w in gated if per_new[w] < per_base[w] - 0.5)
            n10 = sum(1 for w in wid
                      if (per_new.get(w, per_base[w]) if w in per_new else per_base[w]) > 10)
            res.append(dict(src=src, gap=gap, lam=lam, overall=ov, d=ov - base_overall,
                            gate=n_gate, help_w=help_, hurt_w=hurt, over10=n10))
R = pd.DataFrame(res).sort_values("overall")
print(f"基準: overall={base_overall:.4f} / >10={sum(1 for w in wid if per_base[w] > 10)}")
print(R.head(15).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for src, grp in R.groupby("src"):
    gbest = grp.groupby("lam")["overall"].min()
    ax[0].plot(gbest.index, gbest.values, "o-", label=src)
ax[0].axhline(base_overall, c="k", ls="--", label="基準(blend)")
ax[0].set_xlabel("λ"); ax[0].set_ylabel("OOF overall"); ax[0].legend(fontsize=8)
ax[0].set_title("λ vs overall(srcごとgap最良)")
for src, grp in R.groupby("src"):
    gbest = grp.groupby("lam")["over10"].min()
    ax[1].plot(gbest.index, gbest.values, "o-", label=src)
ax[1].axhline(sum(1 for w in wid if per_base[w] > 10), c="k", ls="--")
ax[1].set_xlabel("λ"); ax[1].set_ylabel("wells>10"); ax[1].legend(fontsize=8); ax[1].set_title("λ vs >10本数")
b = R.iloc[0]
ax[2].bar(["help(−0.5↓)", "hurt(+0.5↑)", "その他gated"],
          [b["help_w"], b["hurt_w"], b["gate"] - b["help_w"] - b["hurt_w"]])
ax[2].set_title(f"best: {b['src']} gap{b['gap']} λ{b['lam']}  Δ={b['d']:+.4f}")
plt.tight_layout(); plt.show()


In [ ]:
# ===== best ルールの内訳(硬い核での回収 vs 正常井戸の破壊) =====
b = R.iloc[0]
out, n_gate, gated = apply_rule(b["src"], b["gap"], b["lam"])
per_new = {w: rmse((out - tvt_true)[rows_of[w]]) for w in wid}
dd = {w: per_new[w] - per_base[w] for w in gated}

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
xs = [abs(d_ref_of(w, b["src"])) for w in gated]
ys = [dd[w] for w in gated]
cs = ["red" if w in set(HARD27) else "steelblue" for w in gated]
ax[0].scatter(xs, ys, c=cs, s=18, alpha=0.7)
ax[0].axhline(0, c="k", lw=0.5)
ax[0].set_xlabel("|d_ref| [ft]"); ax[0].set_ylabel("per-well RMSE 変化")
ax[0].set_title("ゲート井戸の損益(赤=硬い核)")
hard_gain = sum(dd[w] for w in gated if w in set(HARD27))
rest_gain = sum(dd[w] for w in gated if w not in set(HARD27))
ax[1].bar(["硬い核", "その他"], [hard_gain, rest_gain])
ax[1].set_ylabel("per-well RMSE 変化の合計"); ax[1].set_title("どこで稼ぎ/どこで壊すか")
v = sorted(dd.values())
ax[2].plot(v, np.arange(1, len(v) + 1) / len(v)); ax[2].axvline(0, c="k", lw=0.5)
ax[2].set_xlabel("per-well RMSE 変化"); ax[2].set_ylabel("ECDF"); ax[2].set_title(f"gated {len(gated)}井戸の変化分布")
plt.tight_layout(); plt.show()

print(f"best: src={b['src']} gap={b['gap']} λ={b['lam']} → overall {base_overall:.4f}→{b['overall']:.4f} (Δ{b['d']:+.4f})")
print("悪化させた井戸(+1.0以上):")
for w in sorted(gated, key=lambda w: -dd[w]):
    if dd[w] > 1.0:
        print(f"  {w}: {per_base[w]:.1f} → {per_new[w]:.1f}  (d_ref={d_ref_of(w, b['src']):+.1f})")
rule = dict(src=str(b["src"]), gap_min=float(b["gap"]), gap_max=GAP_MAX, lam=float(b["lam"]),
            oof_overall=float(b["overall"]), base=float(base_overall))
(ART / "vote_hedge_rule.json").write_text(json.dumps(rule, indent=1))
print("保存:", ART / "vote_hedge_rule.json", "(submit統合用)")
